# 01. AG-UI 与 Human-in-the-Loop

Agent 要和人交互，需要第三层协议。AG-UI 定义 agent↔前端的事件流（16 种事件类型）：
`RUN_STARTED` → `TEXT_MESSAGE_START/CONTENT/END` → `TOOL_CALL_START/ARGS/END` → `STATE_SNAPSHOT` → `RUN_FINISHED`。

HITL 的关键认知：**interrupt 不是暂停进程**，而是序列化状态（checkpoint）→ 等待外部信号 → 从 checkpoint 恢复。
这让人类审批可以跨越任意时长（甚至重启服务之后）。

> **检查点**：能画出 interrupt 的时序图：agent → checkpoint → human → resume，标出状态在哪一侧。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()

🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
from pathlib import Path
import subprocess, sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent

SCRIPT = ROOT / 'agents' / 'protocols' / 'scripts' / 'hitl_approval_demo.py'
result = subprocess.run([sys.executable, str(SCRIPT), '--scenario', 'mixed', '--auto-approve', '--verbose'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)

Plan (scripted): [{"action": "read_file", "args": {"path": "draft_email.txt"}}, {"action": "send_email", "args": {"to": "team@corp.example", "subject": "Q3 report"}}, {"action": "delete_file", "args": {"path": "/tmp/draft_email.txt"}}]
▶ executed read_file({"path": "draft_email.txt"}) (risk=low)

⏸️  INTERRUPT — agent wants a HIGH-risk action:
   send_email({"to": "team@corp.example", "subject": "Q3 report"})
   [checkpoint saved: 1 done, 2 pending]
   --auto-approve set → APPROVED (in production a human clicks this)
   ✅ approved — resuming agent
▶ executed send_email({"to": "team@corp.example", "subject": "Q3 report"}) (risk=high)

⏸️  INTERRUPT — agent wants a HIGH-risk action:
   delete_file({"path": "/tmp/draft_email.txt"})
   [checkpoint saved: 2 done, 1 pending]
   --auto-approve set → APPROVED (in production a human clicks this)
   ✅ approved — resuming agent
▶ executed delete_file({"path": "/tmp/draft_email.txt"}) (risk=high)

Done: 3 actions processed.
Rejected by human: 0

T

## 风险分级是 HITL 的前提

| 风险级 | 例子 | 策略 |
|-------|------|------|
| 低 | read_file, search | 自动执行 |
| 中 | write_file（沙箱内） | 自动执行 + 审计日志 |
| 高 | send_email, delete_file, payment | interrupt，必须人工批准 |

分级标准：**可逆性**（能否 undo）× **影响面**（内部 vs 外部世界）。
send_email 不可逆且影响外部 → 永远高风险；read_file 可逆且内部 → 自动。

In [3]:
from pathlib import Path
import subprocess, sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent

SCRIPT = ROOT / 'agents' / 'protocols' / 'scripts' / 'hitl_approval_demo.py'
result = subprocess.run([sys.executable, str(SCRIPT), '--scenario', 'mixed', '--use-api', '--auto-approve', '--verbose'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)

Plan (LLM): [{"action": "send_email", "args": {"to": "team@corp.example", "subject": "Q3 Report Draft"}}, {"action": "delete_file", "args": {"path": "Q3_report_draft.docx"}}]

⏸️  INTERRUPT — agent wants a HIGH-risk action:
   send_email({"to": "team@corp.example", "subject": "Q3 Report Draft"})
   [checkpoint saved: 0 done, 2 pending]
   --auto-approve set → APPROVED (in production a human clicks this)
   ✅ approved — resuming agent
▶ executed send_email({"to": "team@corp.example", "subject": "Q3 Report Draft"}) (risk=high)

⏸️  INTERRUPT — agent wants a HIGH-risk action:
   delete_file({"path": "Q3_report_draft.docx"})
   [checkpoint saved: 1 done, 1 pending]
   --auto-approve set → APPROVED (in production a human clicks this)
   ✅ approved — resuming agent
▶ executed delete_file({"path": "Q3_report_draft.docx"}) (risk=high)

Done: 2 actions processed.
Rejected by human: 0

Takeaway: interrupt = serialize state + wait for external signal + resume.
The agent never 'blocks on input()' 

## 真实 LLM 规划 + 审批观察

API 模式下动作计划由 LLM 生成 — 注意 LLM 可能计划出脚本里没预设的动作组合，
审批层必须对**任意**动作生效（基于风险分级），而不是只拦截已知剧本。

## 练习

1. 去掉 `--auto-approve` 在本地运行，体验交互式审批；输入 `n` 拒绝一个动作，观察 agent 如何继续。
2. 增加风险级「中」：write_file 自动执行但记录审计日志。修改 `RISK` 与 `run_plan` 实现。
3. 设计 rejection feedback：当人类拒绝时，把原因（「金额太大」）反馈给 LLM 让它重新规划。在 `plan_llm` 中实现这个循环。